# Stroke Risk Prediction — Data Preprocessing (V1)
**Objectif** : Préparer un fichier `df_clean.csv` prêt pour le notebook de modélisation.  
---
**Prérequis** : exécuter `1_EDA_Data.ipynb` d'abord → génère `features.json`  
**Sortie** : `df_clean.csv`

---

> **Version V1 — Baseline minimal**  
> - Variables chargées depuis `features.json` — pas de répétition depuis l'EDA  
> - Qualité données vérifiée via `check_data_quality()` — pas de code dupliqué  

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import warnings

import sys
sys.path.insert(0, './src')  # src/ au même niveau que les notebooks

import importlib, config
importlib.reload(config)

from config import TARGET, RANDOM_STATE, DATA_PATH, CLEAN_PATH, check_data_quality, load_features

warnings.filterwarnings('ignore')
print('Libraries loaded.')

Libraries loaded.


## 2. Chargement des variables depuis features.json

Les listes `CATEGORICAL_VARS` et `NUMERICAL_VARS` ont été définies et sauvegardées  par l'EDA. On les charge ici directement.

In [2]:
# Chargement depuis features.json généré par l'EDA
TARGET, CATEGORICAL_VARS, NUMERICAL_VARS, ALL_FEATURES = load_features()

features.json charge :
  target           : stroke
  categorical_vars : 15 variables
  numerical_vars   : 20 variables


## 3. Load Dataset

In [3]:
df = pd.read_csv(DATA_PATH)

# Nettoyage du nom de colonne 'alcohol '
df.columns = df.columns.str.strip()

print(f'Shape : {df.shape[0]} lignes x {df.shape[1]} colonnes')
df.head(3)

Shape : 4603 lignes x 36 colonnes


,stroke,gender,age,Race,Marital status,alcohol,smoke,sleep disorder,Health Insurance,General health condition,...,energy,protein,Carbohydrate,Dietary fiber,Total fat,Total saturated fatty acids,Total monounsaturated fatty acids,Total polyunsaturated fatty acids,Potassium,Sodium
0,0,2,2,5,1,0,0,2,2,3,...,1598,62.78,192.19,10.0,65.64,25.112,24.090,8.543,2887,2969
1,0,2,2,1,1,0,0,1,2,3,...,1547,45.35,256.02,17.0,42.56,13.423,15.389,10.613,2058,2091
2,1,1,2,3,1,1,1,2,1,3,...,2466,81.56,254.49,13.0,103.32,43.295,36.727,15.366,3117,5233


## 4. Vérification Qualité
`check_data_quality()` importée de `config.py` — définie une seule fois, appelée ici  
comme confirmation avant traitement. Pas de code `isnull()` ou `duplicated()` réécrit.

In [4]:
# Confirmation qualité — même fonction que dans l'EDA
is_clean = check_data_quality(df)

if not is_clean:
    raise ValueError('Dataset non propre — corriger avant de continuer.')

  Data Quality Report

Valeurs manquantes : 0
  Aucune valeur manquante.

Doublons : 0
  Aucun doublon.

Shape : 4603 lignes x 36 colonnes


## 5. Sauvegarde — df_clean.csv

On reconstruit le dataframe complet (train + test) et on sauvegarde.  
Le Modeling rechargera `df_clean.csv` et refera le split avec le même `random_state`.

In [5]:
# Assertions finales avant sauvegarde
assert df.isnull().sum().sum() == 0
assert df.duplicated().sum()   == 0
assert TARGET in df.columns
assert "alcohol " not in df.columns

df.to_csv(CLEAN_PATH, index=False)

vc  = df[TARGET].value_counts()
pct = df[TARGET].value_counts(normalize=True) * 100

print("=== Sauvegarde ===")
print("Fichier  :", CLEAN_PATH)
print("Shape    :", df.shape)
print("No stroke (0) :", vc[0], f"({pct[0]:.2f}%)")
print("Stroke    (1) :", vc[1], f"({pct[1]:.2f}%)")
print("Ratio         :", round(vc[0]/vc[1], 1), ": 1")

=== Sauvegarde ===
Fichier  : ../dataset/data_clean/df_clean.csv
Shape    : (4603, 36)
No stroke (0) : 4241 (92.14%)
Stroke    (1) : 362 (7.86%)
Ratio         : 11.7 : 1
